# LD-2 — LINE-DETR complet (géométrie + CTC, λ = 1/6) et évaluation unique pré-enregistrée (GPU)

In [ ]:

import time
T_START = time.monotonic()          # pris au tout début de la cellule, avant la copie du kit
import datetime, faulthandler, hashlib, json, os, shlex, shutil, subprocess, sys
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld2 start', _utc(), flush=True)
OUT = Path('/kaggle/working')
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld2-kit', '/kaggle/input/datasets/mabaku/babaorum-ld2-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
CKPT_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/logs/checkpoint_last.pt', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/logs/checkpoint_last.pt']
LD1_CANDIDATES = ['/kaggle/input/babaorum-ld1-run/ld1/ld1_step001871.pt', '/kaggle/input/notebooks/mabaku/babaorum-ld1-run/ld1/ld1_step001871.pt']
REF_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/e2e5_lambda_run_report.json', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/e2e5_lambda_run_report.json']
REF_PREFIX = '3abfe0d7'
EST_FACTOR = 1.5
INIT_SHA = 'ef5e826803c347f73e49c83c0f21f61399f9ea06ee985905bd17644dff3844fa'
INIT_STEP = 2000
DEC_SHA = 'a1c5fbab288c99e4911789f926d2d568815fd2a165310da5020ac7c140f8197b'
DEC_STEP = 1871
TIMEOUTS = {'D': 900, 'T': 2700, 'E': 9000, 'EW': 4800, 'FREEZE': 300, 'CAL': 900, 'pytest_heavy': 5400, 'TW': 1500}
LIMITS = {'deadline_s': 19800, 'tail_reserve_s': 2400, 'stop_margin_s': 120, 'eval_margin_s': 300, 'ew_grace_s': 180}
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld2.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
CKPT = resolve_file(CKPT_CANDIDATES)
LD1CK = resolve_file(LD1_CANDIDATES)
assert sha256(CKPT) == INIT_SHA, "checkpoint de l'encodeur (E2E-5) : sha différent"
assert sha256(LD1CK) == DEC_SHA, "checkpoint du décodeur (LD-1) : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)
import ld2_core as C2

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "train_ld2.py"), "--phase", phase,
            "--root", str(PAGES), "--init-checkpoint", str(CKPT), "--decoder-checkpoint", str(LD1CK),
            "--device", "cuda", "--cache-dir", "/tmp/ld2_hf", "--out", str(RUN), *extra]

def wrapped(ph, argv, inner_s):
    """Étapes du témoin (TW, EW) : le code de sortie va dans <RUN>/../exits/ld2_<ph>_rc.txt et l'étape rend TOUJOURS 0 au carnet,
    même après un arrêt dur du processus (OOM de l'hôte, segfault). `timeout -s KILL inner_s` tue la commande AVANT le délai de
    l'étape du carnet (inner_s entre le chien de garde et ce délai), pour que sh survive, écrive 137 et rende 0 ; FREEZE, CAL
    et EVAL relisent ce fichier."""
    exits = RUN.parent / "exits"
    exits.mkdir(parents=True, exist_ok=True)
    rc = shlex.quote(str(exits / f"ld2_{ph}_rc.txt"))
    cmd = shlex.join(["timeout", "-s", "KILL", f"{inner_s:.1f}", *[str(a) for a in argv]])
    return ["sh", "-c", cmd + f"; echo $? > {rc}; exit 0"]

REF = resolve_file(REF_CANDIDATES)
RUN = OUT / "ld2"
CAL_OUT = OUT / "ld2_cal"
EXITS = OUT / "exits"
TAIL = LIMITS["tail_reserve_s"] + LIMITS["stop_margin_s"]       # gardé pour FREEZE, CAL et EVAL
class Incomplete(Exception):
    pass

def remaining():
    return LIMITS["deadline_s"] - (time.monotonic() - T_START)

def allowed(cap, reserve=0.0):
    """max(0, min(plafond, restant - réserve))."""
    return C2.allowed(cap, remaining(), reserve)

def step_argv(ph, build_argv, cap, reserve=0.0, **kw):
    """Étape argv dont le délai est calculé au lancement ; délai nul : INCOMPLETE sans lancer l'étape."""
    def argv():
        t = allowed(cap, reserve)
        if t <= 0:
            raise Incomplete(f"échéance globale atteinte avant l'étape {ph}")
        st["timeout_s"] = t
        return build_argv()
    st = {"name": ph, "argv": argv, "timeout_s": cap, **kw}
    return st

def tw_argv():
    """Chronométrage du témoin : budget propre ; sans temps, il n'est PAS lancé (S_t reste 0, témoin NOT_RUN)."""
    t = allowed(TIMEOUTS["TW"], TAIL)
    if t < 60:
        TW_STEP["timeout_s"] = 60
        return [sys.executable, "-c", "pass"]
    TW_STEP["timeout_s"] = t + LIMITS["ew_grace_s"]
    return wrapped("TW", _argv("TW", ["--deadline-s", str(t)]), t + 0.75 * LIMITS["ew_grace_s"])

def ew_argv():
    """Témoin : budget propre ; s'il n'y a plus de temps, il n'est PAS lancé (état NOT_RUN) et le principal continue."""
    t = allowed(TIMEOUTS["EW"], TAIL)
    if t < 60:
        man = json.loads((RUN / "ld2_manifest.json").read_text())
        (RUN / "ld2_witness_status.json").write_text(json.dumps(
            {"status": "NOT_RUN", "reason": "échéance globale", "S2": man["S2"], "S_t": man["S_t"]}, indent=1))
        EW_STEP["timeout_s"] = 60
        return [sys.executable, "-c", "pass"]
    EW_STEP["timeout_s"] = t + LIMITS["ew_grace_s"]
    return wrapped("EW", _argv("EW", ["--deadline-s", str(t)]), t + 0.75 * LIMITS["ew_grace_s"])

def s_estimate():
    """Porte avant EVAL : estimation = chargements + n pages x durée mesurée par CAL sur une page d'ENTRAÎNEMENT. Si elle
    dépasse le temps restant moins la marge : INCOMPLETE, avant d'ouvrir une page d'évaluation."""
    cal = json.loads((CAL_OUT / "ld2_cal.json").read_text())
    avail = remaining() - LIMITS["eval_margin_s"]
    est = cal["estimate_s"] * EST_FACTOR                       # l'estimation de CAL (une page d'entraînement) est majorée
    (OUT / "ld2_eval_estimate.json").write_text(json.dumps(
        {"estimate_s": est, "factor": EST_FACTOR, "remaining_s": remaining(), "margin_s": LIMITS["eval_margin_s"], "available_s": avail,
         "cal": cal}, indent=1))
    print(f"ESTIMATION_EVAL_S {est:.0f} ; disponible {avail:.0f}", flush=True)
    if est > avail:
        raise Incomplete(f"budget insuffisant : estimation {est:.0f} s > {avail:.0f} s restants")

def eval_argv():
    t = allowed(LIMITS["deadline_s"], LIMITS["eval_margin_s"])
    if t <= 0:
        raise Incomplete("échéance globale atteinte avant EVAL")
    EVAL_STEP["timeout_s"] = t
    return _argv("EVAL", ["--reference-report", str(REF)])

def ld2_cleanup(rec):
    """TIMEOUT, échéance ou échec d'étape : INCOMPLETE à l'étape, sauf si l'étape a écrit FAILED (device, refus, garde)."""
    if rec["status"] == "TIMEOUT" or str(rec.get("exception") or "").startswith("Incomplete:"):
        rec["status"] = "INCOMPLETE"
    elif rec["status"] == "FAILED" and rec["exit_code"] != "EXCEPTION":
        f = EXITS / f"ld2_run_exit_{rec['step']}.json"
        if not f.is_file() or json.loads(f.read_text()).get("status") != "FAILED":
            rec["status"] = "INCOMPLETE"

def s_controls():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA indisponible : run complet refusé")
    assert sha256(REF).startswith(REF_PREFIX), "rapport de référence B3 + λ : sha différent"     # aucune page lue
    from importlib.metadata import version as _v
    (OUT / "ld2_env.json").write_text(json.dumps({     # échec (PackageNotFoundError) si transformers ou huggingface_hub absents
        "python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda, "numpy": _v("numpy"),
        "PIL": _v("pillow"), "transformers": _v("transformers"), "huggingface_hub": _v("huggingface_hub")}, indent=1))
    from babaorum.experimental.line_detr import LineDetr, load_enc_det_from_e2e5
    ck = torch.load(CKPT, map_location="cpu")
    load_enc_det_from_e2e5(LineDetr(), ck["model"], expected_step=INIT_STEP, step=ck["step"])    # init_check de l'encodeur
    ck1 = torch.load(LD1CK, map_location="cpu")
    want = LineDetr().state_dict()                                                              # init_check du décodeur de LD-1
    assert ck1["step"] == DEC_STEP, "décodeur de LD-1 : pas différent"
    assert set(ck1["model"]) == set(want) and all(tuple(ck1["model"][k].shape) == tuple(v.shape) for k, v in want.items()), \
        "décodeur de LD-1 : clés ou formes différentes"
    print("controls ok", _utc(), flush=True)

OFFLINE = lambda: dict(os.environ, HF_HUB_OFFLINE="1")       # hors ligne sauf pendant D
D_STEP = step_argv("D", lambda: _argv("D"), TIMEOUTS["D"])
T_STEP = step_argv("T", lambda: _argv("T"), TIMEOUTS["T"], env=OFFLINE)
TW_STEP = {"name": "TW", "argv": tw_argv, "timeout_s": TIMEOUTS["TW"], "env": OFFLINE}
E_STEP = step_argv("E", lambda: _argv("E"), TIMEOUTS["E"], TAIL, env=OFFLINE)
EW_STEP = {"name": "EW", "argv": ew_argv, "timeout_s": TIMEOUTS["EW"], "env": OFFLINE}
FREEZE_STEP = step_argv("FREEZE", lambda: _argv("FREEZE"), TIMEOUTS["FREEZE"], env=OFFLINE)
CAL_STEP = step_argv("CAL", lambda: _argv("CAL", ["--cal-out", str(CAL_OUT)]), TIMEOUTS["CAL"], env=OFFLINE)
EVAL_STEP = {"name": "EVAL", "argv": eval_argv, "timeout_s": LIMITS["deadline_s"], "env": OFFLINE}
STEPS = [{"name": "controls", "fn": s_controls}, D_STEP, T_STEP, TW_STEP, E_STEP, EW_STEP, FREEZE_STEP, CAL_STEP,
         {"name": "estimate", "fn": s_estimate}, EVAL_STEP]
EXIT = None
try:
    EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=60.0, cleanup=ld2_cleanup)
finally:
    (OUT / "ld2_run_exit.json").write_text(json.dumps(EXIT if EXIT is not None else {"status": "FAILED_BEFORE_EXIT"},
                                                      indent=1, default=str))
print("EXIT", json.dumps({k: EXIT[k] for k in ("status", "step", "exit_code")}) if EXIT else "none", flush=True)
if (RUN / "ld2_report.json").is_file():
    print("REPORT_SHA256", sha256(RUN / "ld2_report.json"), flush=True)
